In [1]:
import os
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import kagglehub


In [2]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

print("Experiment: ResNet50 Focal Loss + 5-Fold CV")
print("Images: Original")
print("Focal alpha: 0.29")
print("Focal gamma: 2.0")
print("Dropout: Disabled")
print("Balanced batches: Disabled")
print("Class weights: Disabled")

TensorFlow version: 2.20.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Experiment: ResNet50 Focal Loss + 5-Fold CV
Images: Original
Focal alpha: 0.29
Focal gamma: 2.0
Dropout: Disabled
Balanced batches: Disabled
Class weights: Disabled


In [3]:
dataset_path = kagglehub.dataset_download(
    "aryansinghal10/alzheimers-multiclass-dataset-equal-and-augmented"
)

image_root = Path(dataset_path) / "combined_images"

print("Dataset path:", dataset_path)
print("Image folder:", image_root)
print("Image folder exists:", image_root.exists())

Mounting files to /kaggle/input/datasets/aryansinghal10/alzheimers-multiclass-dataset-equal-and-augmented...
Dataset path: /kaggle/input/datasets/aryansinghal10/alzheimers-multiclass-dataset-equal-and-augmented
Image folder: /kaggle/input/datasets/aryansinghal10/alzheimers-multiclass-dataset-equal-and-augmented/combined_images
Image folder exists: True


In [4]:
class_mapping = {
    "NonDemented": 0,
    "VeryMildDemented": 1,
    "MildDemented": 1,
    "ModerateDemented": 1
}

records = []

for class_name, binary_label in class_mapping.items():
    class_folder = image_root / class_name

    for file_path in sorted(class_folder.iterdir()):
        if file_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            records.append({
                "filepath": str(file_path),
                "original_class": class_name,
                "label": binary_label
            })

df = pd.DataFrame(records)

print("Total images:", len(df))

print("\nBinary class counts:")
print(df["label"].value_counts().sort_index())

print("\nOriginal class counts:")
print(df["original_class"].value_counts())

Total images: 44000

Binary class counts:
label
0    12800
1    31200
Name: count, dtype: int64

Original class counts:
original_class
NonDemented         12800
VeryMildDemented    11200
MildDemented        10000
ModerateDemented    10000
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

initial_train_df, temporary_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

initial_validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temporary_df["label"]
)

development_df = pd.concat(
    [initial_train_df, initial_validation_df],
    ignore_index=True
)

development_df = development_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Development images:", len(development_df))
print(development_df["label"].value_counts().sort_index())

print("\nUntouched test images:", len(test_df))
print(test_df["label"].value_counts().sort_index())

Development images: 37400
label
0    10880
1    26520
Name: count, dtype: int64

Untouched test images: 6600
label
0    1920
1    4680
Name: count, dtype: int64


In [6]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_original_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    return image, tf.cast(label, tf.float32)

def create_dataset(dataframe, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((
        dataframe["filepath"].values,
        dataframe["label"].values
    ))

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_original_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    return dataset.prefetch(AUTOTUNE)

test_dataset = create_dataset(test_df)

print("Original-image pipeline created.")
print("Test dataset batches:", len(test_dataset))

Original-image pipeline created.
Test dataset batches: 207


I0000 00:00:1789401782.341558      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [7]:
image_batch, label_batch = next(iter(test_dataset))

print("Image batch shape:", image_batch.shape)
print("Label batch shape:", label_batch.shape)

print(
    "Pixel range:",
    float(tf.reduce_min(image_batch)),
    "to",
    float(tf.reduce_max(image_batch))
)

print("First labels:", label_batch[:10].numpy())

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Pixel range: 0.0 to 255.0
First labels: [1. 1. 1. 1. 1. 1. 1. 0. 1. 1.]


In [8]:
from sklearn.model_selection import StratifiedKFold

N_SPLITS = 5

stratified_kfold = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

fold_splits = list(
    stratified_kfold.split(
        development_df["filepath"],
        development_df["label"]
    )
)

for fold_number, (train_indices, val_indices) in enumerate(
    fold_splits,
    start=1
):
    fold_train_labels = development_df.iloc[
        train_indices
    ]["label"]

    fold_val_labels = development_df.iloc[
        val_indices
    ]["label"]

    print(
        f"Fold {fold_number}: "
        f"Training = {len(train_indices)}, "
        f"Validation = {len(val_indices)}"
    )

    print(
        "  Train counts:",
        fold_train_labels.value_counts().sort_index().to_dict()
    )
    print(
        "  Validation counts:",
        fold_val_labels.value_counts().sort_index().to_dict()
    )

Fold 1: Training = 29920, Validation = 7480
  Train counts: {0: 8704, 1: 21216}
  Validation counts: {0: 2176, 1: 5304}
Fold 2: Training = 29920, Validation = 7480
  Train counts: {0: 8704, 1: 21216}
  Validation counts: {0: 2176, 1: 5304}
Fold 3: Training = 29920, Validation = 7480
  Train counts: {0: 8704, 1: 21216}
  Validation counts: {0: 2176, 1: 5304}
Fold 4: Training = 29920, Validation = 7480
  Train counts: {0: 8704, 1: 21216}
  Validation counts: {0: 2176, 1: 5304}
Fold 5: Training = 29920, Validation = 7480
  Train counts: {0: 8704, 1: 21216}
  Validation counts: {0: 2176, 1: 5304}


In [9]:
MAX_EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 0.0001

FOCAL_ALPHA = 0.29
FOCAL_GAMMA = 2.0

print("Number of folds:", N_SPLITS)
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Maximum epochs per fold:", MAX_EPOCHS)
print("Early-stopping patience:", PATIENCE)
print("Learning rate:", LEARNING_RATE)
print("Focal alpha:", FOCAL_ALPHA)
print("Focal gamma:", FOCAL_GAMMA)

print("Preprocessing: Original images")
print("Data augmentation: Disabled")
print("Dropout rate: Disabled")
print("Natural batches: Enabled")
print("Class weights: Disabled")

Number of folds: 5
Image size: (224, 224)
Batch size: 32
Maximum epochs per fold: 30
Early-stopping patience: 5
Learning rate: 0.0001
Focal alpha: 0.29
Focal gamma: 2.0
Preprocessing: Original images
Data augmentation: Disabled
Dropout rate: Disabled
Natural batches: Enabled
Class weights: Disabled


In [10]:
def build_focal_resnet50():
    tf.keras.utils.set_random_seed(SEED)

    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))

    x = tf.keras.applications.resnet50.preprocess_input(inputs)

    base_model = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(*IMG_SIZE, 3)
    )

    base_model.trainable = False

    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    model = tf.keras.Model(
        inputs,
        outputs,
        name="ResNet50_Focal_CV"
    )

    focal_loss = tf.keras.losses.BinaryFocalCrossentropy(
        apply_class_balancing=True,
        alpha=FOCAL_ALPHA,
        gamma=FOCAL_GAMMA,
        from_logits=False
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss=focal_loss,
        metrics=["accuracy"]
    )

    return model

preview_model = build_focal_resnet50()

print("Total parameters:", preview_model.count_params())
print(
    "Trainable parameters:",
    sum(tf.keras.backend.count_params(weight)
        for weight in preview_model.trainable_weights)
)
print("ResNet50 frozen: True")
print("Dropout enabled: False")

preview_model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Total parameters: 23589761
Trainable parameters: 2049
ResNet50 frozen: True
Dropout enabled: False


Model: "ResNet50_Focal_CV"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ input_layer[0][0] │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ input_layer[0][0] │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │      2,049 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,589,761 (89.99 MB)

 Trainable params: 2,049 (8.00 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [11]:
def create_fold_callbacks(fold_number):
    model_path = (
        f"/kaggle/working/"
        f"resnet50_focal_cv_fold_{fold_number}_best.keras"
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1
        ),

        tf.keras.callbacks.ModelCheckpoint(
            model_path,
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        )
    ]

    return callbacks, model_path

print("Fold callback function created.")

Fold callback function created.


In [ ]:
del preview_model
tf.keras.backend.clear_session()
gc.collect()

fold_history_frames = []
fold_summaries = []
fold_model_paths = []

for fold_number, (train_indices, val_indices) in enumerate(
    fold_splits,
    start=1
):
    print(
        f"\n{'=' * 20} "
        f"FOLD {fold_number}/{N_SPLITS} "
        f"{'=' * 20}"
    )

    fold_train_df = development_df.iloc[
        train_indices
    ].reset_index(drop=True)

    fold_val_df = development_df.iloc[
        val_indices
    ].reset_index(drop=True)

    fold_train_dataset = create_dataset(
        fold_train_df,
        training=True
    )

    fold_val_dataset = create_dataset(
        fold_val_df,
        training=False
    )

    tf.keras.backend.clear_session()
    gc.collect()

    model = build_focal_resnet50()

    fold_callbacks, model_path = create_fold_callbacks(
        fold_number
    )

    history = model.fit(
        fold_train_dataset,
        validation_data=fold_val_dataset,
        epochs=MAX_EPOCHS,
        callbacks=fold_callbacks,
        verbose=1
    )

    history_df = pd.DataFrame(history.history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
    history_df.insert(0, "fold", fold_number)

    fold_history_frames.append(history_df)
    fold_model_paths.append(model_path)

    best_index = int(
        np.argmin(history.history["val_loss"])
    )

    fold_summaries.append({
        "Fold": fold_number,
        "Epochs Trained": len(history.history["loss"]),
        "Best Epoch": best_index + 1,
        "Best Val Loss": history.history["val_loss"][best_index],
        "Val Accuracy at Best Loss":
            history.history["val_accuracy"][best_index],
        "Highest Val Accuracy":
            max(history.history["val_accuracy"])
    })

    del model
    del fold_train_dataset
    del fold_val_dataset
    gc.collect()

all_fold_history = pd.concat(
    fold_history_frames,
    ignore_index=True
)

fold_summary_df = pd.DataFrame(fold_summaries)

all_fold_history.to_csv(
    "/kaggle/working/resnet50_focal_cv_history.csv",
    index=False
)

fold_summary_df.to_csv(
    "/kaggle/working/resnet50_focal_cv_fold_summary.csv",
    index=False
)

display(fold_summary_df)
print("All five folds finished and saved.")